In [1]:
# input
test_file = "../../../../data/dataset/metalnet_test_ched.tsv"
train_file = "../../../../data/dataset/metalnet_train_ched.tsv"
pred_file = "./tmp/denovo_mbp_ids_pred.tsv"
mbp_denovo_ids_file = "./tmp/denovo_mbp_ids.tsv"

In [2]:
import pandas as pd

df_train = pd.read_table(train_file)
df_test = pd.read_table(test_file)
df_test['data_type'] = "test"
df_train['data_type'] = "train"
df = pd.concat([df_train, df_test])

dn_ids = set(pd.read_table(mbp_denovo_ids_file, header=None)[0])
df_denovo = df[df['seq_id'].map(lambda x: x in dn_ids)]
len(dn_ids)
len(df_denovo)

13

240

### analysis

In [3]:
df_pred = pd.read_table(pred_file)

In [4]:
def get_anno_residues(df, data_type: str | None):
    df_ = df
    if data_type is not None:
        assert data_type in ['train', 'test']
        df_ = df[df['data_type'] == data_type]

    residues = set()
    for _, row in df_.iterrows():
        residues.add((row['seq_id'], row['resi_seq_posi']))
    return residues

def get_pred_residues(df):
    residues = set()
    for _, row in df.iterrows():
        for i in row['posi'].split(","):
            residues.add((row['seq_id'], int(i)))
    return residues

def calc_metrics(anno, pred):
    inter = anno & pred
    prec = len(inter) / len(pred)
    reca = len(inter) / len(anno)
    f1 = 2 * prec * reca / (prec + reca)
    return prec, reca, f1


train_anno_residues = get_anno_residues(df_denovo, "train")
test_anno_residues = get_anno_residues(df_denovo, "test")

train_pred_residues = get_pred_residues(pd.merge(df_train, df_pred, on="seq_id"))
test_pred_residues = get_pred_residues(pd.merge(df_test, df_pred, on="seq_id"))

len(train_anno_residues)
len(test_anno_residues)
len(train_pred_residues)
len(test_pred_residues)

calc_metrics(train_anno_residues, train_pred_residues)
calc_metrics(test_anno_residues, test_pred_residues)

225

15

38

1

(1.0, 0.1688888888888889, 0.2889733840304183)

(1.0, 0.06666666666666667, 0.125)